# EDA de MovieLens ml-latest-small

Este cuaderno responde:

1. ¿Qué datos tengo?
2. ¿Hay datos faltantes?
3. ¿Existen valores atípicos?
4. ¿Cómo se distribuyen los datos?
5. ¿Qué relaciones existen entre variables?

Coloca los archivos `movies.csv`, `ratings.csv`, `tags.csv` y `links.csv` en la carpeta `ml-latest-small`.

In [ ]:
# Si hace falta instalar dependencias, descomenta la siguiente línea:
# %pip install pandas matplotlib seaborn

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

DATA_DIR = Path('./ml-latest-small')
OUTPUT_DIR = Path('./output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Datos:', DATA_DIR.resolve())
print('Salida:', OUTPUT_DIR.resolve())

## Carga y preparación de datos

In [ ]:
def load_data(data_dir):
    filenames = {
        'movies': 'movies.csv',
        'ratings': 'ratings.csv',
        'tags': 'tags.csv',
        'links': 'links.csv'
    }
    result = {}
    for name, filename in filenames.items():
        path = data_dir / filename
        if not path.exists():
            raise FileNotFoundError(f'No se encontró: {path}')
        result[name] = pd.read_csv(path, encoding='utf-8')
    return result


def save_plot(filename, title):
    plt.title(title, fontsize=14, fontweight='bold', pad=12)
    plt.tight_layout()
    path = OUTPUT_DIR / filename
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print('Guardado:', path)


data = load_data(DATA_DIR)
movies = data['movies'].copy()
ratings = data['ratings'].copy()
tags = data['tags'].copy()
links = data['links'].copy()

for frame in (ratings, tags):
    frame['datetime'] = pd.to_datetime(frame['timestamp'], unit='s', errors='coerce', utc=True)
    frame['year'] = frame['datetime'].dt.year

tags['tag_normalized'] = tags['tag'].fillna('').astype(str).str.strip().str.lower()
tags['tag_valid'] = tags['tag_normalized'].ne('')
valid_tags = tags.loc[tags['tag_valid']].copy()

movies['genres'] = movies['genres'].fillna('')
movies['genre_count'] = movies['genres'].apply(
    lambda value: 0 if value in ('', '(no genres listed)') else len(value.split('|'))
)

print('Archivos cargados correctamente.')

## 1. ¿Qué datos tengo?

In [ ]:
descriptions = {
    'movies': 'Información de películas y géneros',
    'ratings': 'Calificaciones dadas por usuarios',
    'tags': 'Etiquetas asignadas por usuarios',
    'links': 'Enlaces externos de películas'
}

rows = []
for name, frame in data.items():
    rows.append({
        'archivo': f'{name}.csv',
        'descripción': descriptions[name],
        'filas': len(frame),
        'columnas': len(frame.columns),
        'variables': ', '.join(frame.columns)
    })

data_summary = pd.DataFrame(rows)
data_summary.to_csv(OUTPUT_DIR / 'data_summary.csv', index=False, encoding='utf-8')
display(data_summary)

print(f'El conjunto contiene {len(ratings):,} calificaciones, {len(valid_tags):,} etiquetas válidas y {len(movies):,} películas.')

## 2. ¿Hay datos faltantes?

In [ ]:
missing_rows = []
for name, frame in data.items():
    for column in frame.columns:
        missing_count = int(frame[column].isna().sum())
        missing_percent = missing_count / len(frame) * 100 if len(frame) else 0
        empty_count = 0
        if pd.api.types.is_object_dtype(frame[column]):
            empty_count = int(frame[column].fillna('').astype(str).str.strip().eq('').sum())
        missing_rows.append({
            'archivo': f'{name}.csv',
            'columna': column,
            'faltantes': missing_count,
            'porcentaje_faltante': round(missing_percent, 4),
            'cadenas_vacias': empty_count
        })

missing_report = pd.DataFrame(missing_rows)
missing_report.to_csv(OUTPUT_DIR / 'missing_values.csv', index=False, encoding='utf-8')
problems = missing_report.query('faltantes > 0 or cadenas_vacias > 0')

if problems.empty:
    print('No se encontraron valores faltantes ni campos vacíos.')
else:
    display(problems)

## 3. ¿Existen valores atípicos?

Se aplica el método del rango intercuartílico (IQR).

In [ ]:
def calculate_iqr(series):
    numeric = pd.to_numeric(series, errors='coerce').dropna()
    if numeric.empty:
        return {
            'q1': 0, 'q3': 0, 'iqr': 0,
            'limite_inferior': 0, 'limite_superior': 0,
            'atipicos': 0, 'porcentaje_atipicos': 0
        }
    q1 = numeric.quantile(0.25)
    q3 = numeric.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (numeric < lower) | (numeric > upper)
    return {
        'q1': q1, 'q3': q3, 'iqr': iqr,
        'limite_inferior': lower, 'limite_superior': upper,
        'atipicos': int(mask.sum()),
        'porcentaje_atipicos': mask.mean() * 100
    }

tag_frequency = valid_tags.groupby('tag_normalized').size()
variables = {
    'Valor de rating': ratings['rating'],
    'Ratings por usuario': ratings.groupby('userId').size(),
    'Ratings por película': ratings.groupby('movieId').size(),
    'Tags por usuario': valid_tags.groupby('userId').size(),
    'Tags por película': valid_tags.groupby('movieId').size(),
    'Usos por etiqueta': tag_frequency,
    'Géneros por película': movies['genre_count']
}

outlier_rows = []
for name, values in variables.items():
    outlier_rows.append({'variable': name, **calculate_iqr(values)})

outlier_report = pd.DataFrame(outlier_rows)
outlier_report.to_csv(OUTPUT_DIR / 'outliers_iqr.csv', index=False, encoding='utf-8')
display(outlier_report.round(2))

plt.figure(figsize=(11, 5))
sns.boxplot(x=tag_frequency.values, color='darkcyan')
plt.xlabel('Número de usos')
save_plot('outliers_tag_frequency.png', 'Valores atípicos en la frecuencia de etiquetas')

## 4. ¿Cómo se distribuyen los datos?

In [ ]:
tag_counts = valid_tags['tag_normalized'].value_counts()

plt.figure(figsize=(12, 8))
tag_counts.head(20).sort_values().plot(kind='barh', color='darkcyan')
plt.xlabel('Número de usos')
plt.ylabel('Etiqueta')
save_plot('top_20_tags.png', 'Top 20 etiquetas más utilizadas')

plt.figure(figsize=(10, 6))
sns.histplot(tag_counts, bins=30, kde=True, color='steelblue')
plt.xlabel('Usos por etiqueta')
plt.ylabel('Cantidad de etiquetas')
save_plot('distribution_tag_frequency.png', 'Distribución de usos por etiqueta')

plt.figure(figsize=(10, 6))
sns.histplot(ratings['rating'], discrete=True, color='seagreen')
plt.xlabel('Calificación')
plt.ylabel('Cantidad de calificaciones')
save_plot('distribution_ratings.png', 'Distribución de calificaciones')

ratings_by_year = ratings.dropna(subset=['year']).groupby('year').size()
plt.figure(figsize=(11, 6))
ratings_by_year.plot(kind='bar', color='slateblue')
plt.xlabel('Año')
plt.ylabel('Cantidad de ratings')
save_plot('ratings_by_year.png', 'Cantidad de ratings por año')

tags_by_year = valid_tags.dropna(subset=['year']).groupby('year').size()
plt.figure(figsize=(11, 6))
tags_by_year.plot(kind='bar', color='darkorange')
plt.xlabel('Año')
plt.ylabel('Cantidad de tags')
save_plot('tags_by_year.png', 'Cantidad de tags por año')

genre_counts = {}
for genres in movies['genres']:
    if genres and genres != '(no genres listed)':
        for genre in genres.split('|'):
            genre_counts[genre] = genre_counts.get(genre, 0) + 1

genre_series = pd.Series(genre_counts).sort_values()
plt.figure(figsize=(10, 7))
genre_series.plot(kind='barh', color='mediumpurple')
plt.xlabel('Cantidad de películas')
plt.ylabel('Género')
save_plot('movie_genres.png', 'Distribución de películas por género')

print('Resumen de etiquetas:')
print(f'Aplicaciones totales: {len(tags):,}')
print(f'Aplicaciones válidas: {len(valid_tags):,}')
print(f'Etiquetas únicas: {tag_counts.size:,}')
print(f'Promedio de usos por etiqueta: {tag_counts.mean():.2f}')
print(f'Mediana de usos por etiqueta: {tag_counts.median():.2f}')
print(f'Rating promedio: {ratings["rating"].mean():.2f}')
print(f'Rating mediano: {ratings["rating"].median():.2f}')

## 5. ¿Qué relaciones existen entre variables?

In [ ]:
ratings_per_user = ratings.groupby('userId').size().sort_values(ascending=False)
tags_per_user = valid_tags.groupby('userId').size().sort_values(ascending=False)
ratings_per_movie = ratings.groupby('movieId').size().sort_values(ascending=False)
tags_per_movie = valid_tags.groupby('movieId').size().sort_values(ascending=False)

activity = pd.concat([
    ratings_per_user.rename('ratings_count'),
    tags_per_user.rename('tags_count')
], axis=1).fillna(0)
activity.index.name = 'userId'
activity.reset_index().to_csv(OUTPUT_DIR / 'user_activity.csv', index=False, encoding='utf-8')

movie_activity = pd.concat([
    ratings_per_movie.rename('ratings_count'),
    tags_per_movie.rename('tags_count')
], axis=1).fillna(0)
movie_activity.index.name = 'movieId'
movie_activity.reset_index().to_csv(OUTPUT_DIR / 'movie_activity.csv', index=False, encoding='utf-8')

correlation = activity['ratings_count'].corr(activity['tags_count'])

plt.figure(figsize=(10, 6))
sns.scatterplot(data=activity, x='ratings_count', y='tags_count', alpha=0.7, color='crimson')
plt.xlabel('Ratings realizados por usuario')
plt.ylabel('Tags creados por usuario')
save_plot('relationship_user_activity.png', 'Relación entre ratings y tags por usuario')

print(f'Usuarios con ratings: {ratings["userId"].nunique():,}')
print(f'Usuarios con tags: {valid_tags["userId"].nunique():,}')
print(f'Películas con ratings: {ratings["movieId"].nunique():,}')
print(f'Películas con tags: {valid_tags["movieId"].nunique():,}')
print(f'Correlación ratings-tags por usuario: {correlation:.4f}')

display(ratings_per_user.head(10).rename('ratings').reset_index())
display(ratings_per_movie.head(10).rename('ratings').reset_index())
display(tags_per_user.head(10).rename('tags').reset_index())
display(tags_per_movie.head(10).rename('tags').reset_index())

## Generar informe legible

In [ ]:
def add_top_items(lines, title, series, item_label):
    lines.append(title)
    for position, (item, value) in enumerate(series.head(10).items(), start=1):
        lines.append(f'  {position:>2}. {item_label}: {item} | Cantidad: {value:,.0f}')


report_lines = []
report_lines.append('INFORME DE ANÁLISIS EXPLORATORIO DE DATOS')
report_lines.append('DATASET: MOVIELENS ML-LATEST-SMALL')
report_lines.append('=' * 90)
report_lines.append('Este informe responde las cinco preguntas principales del EDA.')
report_lines.append('Los valores fueron calculados directamente desde los archivos CSV.')
report_lines.append('')

report_lines.append('=' * 90)
report_lines.append('1. ¿QUÉ DATOS TENGO?')
report_lines.append('=' * 90)
report_lines.append(data_summary.to_string(index=False))
report_lines.append('')
report_lines.append(f'El conjunto contiene {len(ratings):,} calificaciones, {len(valid_tags):,} aplicaciones válidas de etiquetas y {len(movies):,} películas.')
report_lines.append('')

report_lines.append('=' * 90)
report_lines.append('2. ¿HAY DATOS FALTANTES?')
report_lines.append('=' * 90)
if problems.empty:
    report_lines.append('No se encontraron valores faltantes ni campos vacíos.')
else:
    for row in problems.itertuples():
        report_lines.append(f'- {row.archivo} / {row.columna}: {row.faltantes:,} faltantes ({row.porcentaje_faltante:.2f}%) y {row.cadenas_vacias:,} cadenas vacías.')
report_lines.append('')

report_lines.append('=' * 90)
report_lines.append('3. ¿EXISTEN VALORES ATÍPICOS?')
report_lines.append('=' * 90)
for row in outlier_report.itertuples():
    report_lines.append(f'Variable: {row.variable}')
    report_lines.append(f"  ¿Tiene atípicos?: {'Sí' if row.atipicos > 0 else 'No'}")
    report_lines.append(f'  Q1: {row.q1:.2f}')
    report_lines.append(f'  Q3: {row.q3:.2f}')
    report_lines.append(f'  IQR: {row.iqr:.2f}')
    report_lines.append(f'  Cantidad de atípicos: {row.atipicos:,}')
    report_lines.append(f'  Porcentaje de atípicos: {row.porcentaje_atipicos:.2f}%')
    report_lines.append('')

report_lines.append('=' * 90)
report_lines.append('4. ¿CÓMO SE DISTRIBUYEN LOS DATOS?')
report_lines.append('=' * 90)
report_lines.append(f'Aplicaciones totales de tags: {len(tags):,}')
report_lines.append(f'Aplicaciones válidas de tags: {len(valid_tags):,}')
report_lines.append(f'Etiquetas únicas: {tag_counts.size:,}')
report_lines.append(f'Promedio de usos por etiqueta: {tag_counts.mean():.2f}')
report_lines.append(f'Mediana de usos por etiqueta: {tag_counts.median():.2f}')
report_lines.append(f'Promedio de ratings: {ratings["rating"].mean():.2f}')
report_lines.append(f'Mediana de ratings: {ratings["rating"].median():.2f}')
report_lines.append('')

report_lines.append('=' * 90)
report_lines.append('5. ¿QUÉ RELACIONES EXISTEN ENTRE VARIABLES?')
report_lines.append('=' * 90)
report_lines.append(f'Usuarios con ratings: {ratings["userId"].nunique():,}')
report_lines.append(f'Usuarios con tags: {valid_tags["userId"].nunique():,}')
report_lines.append(f'Películas con ratings: {ratings["movieId"].nunique():,}')
report_lines.append(f'Películas con tags: {valid_tags["movieId"].nunique():,}')
report_lines.append('')
add_top_items(report_lines, 'Usuarios con más ratings:', ratings_per_user, 'userId')
report_lines.append('')
add_top_items(report_lines, 'Películas con más ratings:', ratings_per_movie, 'movieId')
report_lines.append('')
add_top_items(report_lines, 'Usuarios con más tags:', tags_per_user, 'userId')
report_lines.append('')
add_top_items(report_lines, 'Películas con más tags:', tags_per_movie, 'movieId')
report_lines.append('')
report_lines.append(f'Correlación entre cantidad de ratings y tags por usuario: {correlation:.4f}')
report_lines.append('Una correlación positiva indica que los usuarios que califican más películas tienden también a crear más etiquetas.')
report_lines.append('')

report_path = OUTPUT_DIR / 'eda_report_legible.txt'
report_path.write_text('\n'.join(report_lines), encoding='utf-8')
print('Informe creado correctamente:')
print(report_path.resolve())
print('')
print('\n'.join(report_lines))

INFORME DE ANÁLISIS EXPLORATORIO DE DATOS
DATASET: MOVIELENS ML-LATEST-SMALL
==========================================================================================
Este informe responde las cinco preguntas principales del EDA.
Los valores fueron calculados directamente desde los archivos CSV.

==========================================================================================
1. ¿QUÉ DATOS TENGO?
==========================================================================================
El dataset contiene información de películas, calificaciones, etiquetas y enlaces externos.

Archivo: movies.csv
  Descripción: Información de películas y géneros
  Filas: 9,742    
  Columnas: 3
  Variables: movieId, title, genres

Archivo: ratings.csv
  Descripción: Calificaciones dadas por usuarios
  Filas: 100,836
  Columnas: 4
  Variables: userId, movieId, rating, timestamp

Archivo: tags.csv
  Descripción: Etiquetas asignadas por usuarios
  Filas: 3,683
  Columnas: 4
  Variables: userId, movieId, tag, timestamp

Archivo: links.csv
  Descripción: Enlaces externos de películas
  Filas: 9,742
  Columnas: 3
  Variables: movieId, imdbId, tmdbId

Interpretación:
  El conjunto contiene 100,836 calificaciones, 3,683 aplicaciones válidas de etiquetas y 9,742 películas.

==========================================================================================
2. ¿HAY DATOS FALTANTES?
==========================================================================================
Se encontraron los siguientes problemas:
  - links.csv / tmdbId: 8 faltantes (0.08%) y 0 cadenas vacías.

Archivo detallado generado: missing_values.csv

==========================================================================================
3. ¿EXISTEN VALORES ATÍPICOS?
==========================================================================================
Se utilizó el método del rango intercuartílico (IQR). Un valor se considera atípico si queda fuera de Q1 - 1.5×IQR o Q3 + 1.5×IQR.

Variable: Valor de rating
  ¿Tiene atípicos?: Sí
  Q1: 3.00
  Q3: 4.00
  IQR: 1.00
  Límite inferior: 1.50
  Límite superior: 5.50
  Cantidad de atípicos: 4,181
  Porcentaje de atípicos: 4.15%

Variable: Ratings por usuario
  ¿Tiene atípicos?: Sí
  Q1: 35.00
  Q3: 168.00
  IQR: 133.00
  Límite inferior: -164.50
  Límite superior: 367.50
  Cantidad de atípicos: 70
  Porcentaje de atípicos: 11.48%

Variable: Ratings por película
  ¿Tiene atípicos?: Sí
  Q1: 1.00
  Q3: 9.00
  IQR: 8.00
  Límite inferior: -11.00
  Límite superior: 21.00
  Cantidad de atípicos: 1,179
  Porcentaje de atípicos: 12.12%

Variable: Tags por usuario
  ¿Tiene atípicos?: Sí
  Q1: 2.25
  Q3: 13.00
  IQR: 10.75
  Límite inferior: -13.88
  Límite superior: 29.12
  Cantidad de atípicos: 12
  Porcentaje de atípicos: 20.69%

Variable: Tags por película
  ¿Tiene atípicos?: Sí
  Q1: 1.00
  Q3: 2.00
  IQR: 1.00
  Límite inferior: -0.50
  Límite superior: 3.50
  Cantidad de atípicos: 225
  Porcentaje de atípicos: 14.31%

Variable: Usos por etiqueta
  ¿Tiene atípicos?: Sí
  Q1: 1.00
  Q3: 2.00
  IQR: 1.00
  Límite inferior: -0.50
  Límite superior: 3.50
  Cantidad de atípicos: 235
  Porcentaje de atípicos: 15.93%

Variable: Géneros por película
  ¿Tiene atípicos?: Sí
  Q1: 1.00
  Q3: 3.00
  IQR: 2.00
  Límite inferior: -2.00
  Límite superior: 6.00
  Cantidad de atípicos: 14
  Porcentaje de atípicos: 0.14%

Archivo detallado generado: outliers_iqr.csv

==========================================================================================
4. ¿CÓMO SE DISTRIBUYEN LOS DATOS?
==========================================================================================
Distribución de etiquetas:
  Aplicaciones totales: 3,683
  Aplicaciones válidas: 3,683
  Etiquetas únicas: 1,475
  Promedio de usos por etiqueta: 2.50
  Mediana de usos por etiqueta: 1.00
  Desviación estándar: 4.70
  Menor frecuencia: 1
  Mayor frecuencia: 131
  Top 10 etiquetas: 358 usos (9.72% del total)
  Top 20 etiquetas: 551 usos (14.96% del total)
  Top 50 etiquetas: 939 usos (25.50% del total)
  Top 100 etiquetas: 1,366 usos (37.09% del total)

Distribución de calificaciones:
  Promedio: 3.50
  Mediana: 3.50
  Mínima: 0.5
  Máxima: 5.0

Interpretación: revise los gráficos generados para observar si las distribuciones son uniformes, concentradas o de cola larga.

==========================================================================================
5. ¿QUÉ RELACIONES EXISTEN ENTRE VARIABLES?
==========================================================================================
Se analizaron relaciones mediante agrupaciones por usuario y película.

Cobertura del dataset:
  Usuarios con ratings: 610
  Usuarios con tags: 58
  Películas con ratings: 9,724
  Películas con tags: 1,572

Usuarios con más ratings:
   1. userId: 414 | Cantidad: 2,698
   2. userId: 599 | Cantidad: 2,478
   3. userId: 474 | Cantidad: 2,108
   4. userId: 448 | Cantidad: 1,864
   5. userId: 274 | Cantidad: 1,346
   6. userId: 610 | Cantidad: 1,302
   7. userId: 68 | Cantidad: 1,260
   8. userId: 380 | Cantidad: 1,218
   9. userId: 606 | Cantidad: 1,115
  10. userId: 288 | Cantidad: 1,055

Películas con más ratings:
   1. movieId: 356 | Cantidad: 329
   2. movieId: 318 | Cantidad: 317
   3. movieId: 296 | Cantidad: 307
   4. movieId: 593 | Cantidad: 279
   5. movieId: 2571 | Cantidad: 278
   6. movieId: 260 | Cantidad: 251
   7. movieId: 480 | Cantidad: 238
   8. movieId: 110 | Cantidad: 237
   9. movieId: 589 | Cantidad: 224
  10. movieId: 527 | Cantidad: 220

Usuarios con más tags:
   1. userId: 474 | Cantidad: 1,507
   2. userId: 567 | Cantidad: 432
   3. userId: 62 | Cantidad: 370
   4. userId: 599 | Cantidad: 323
   5. userId: 477 | Cantidad: 280
   6. userId: 424 | Cantidad: 273
   7. userId: 537 | Cantidad: 100
   8. userId: 125 | Cantidad: 48
   9. userId: 357 | Cantidad: 45
  10. userId: 318 | Cantidad: 41

Películas con más tags:
   1. movieId: 296 | Cantidad: 181
   2. movieId: 2959 | Cantidad: 54
   3. movieId: 924 | Cantidad: 41
   4. movieId: 293 | Cantidad: 35
   5. movieId: 7361 | Cantidad: 34
   6. movieId: 1732 | Cantidad: 32
   7. movieId: 4878 | Cantidad: 29
   8. movieId: 260 | Cantidad: 26
   9. movieId: 79132 | Cantidad: 26
  10. movieId: 135536 | Cantidad: 19

Correlación entre cantidad de ratings y tags por usuario: 0.3579
Interpretación: una correlación positiva indica que los usuarios que califican más películas tienden también a crear más etiquetas.
Nota: userId y movieId son identificadores, por lo que no se debe interpretar su correlación numérica directa.

==========================================================================================
ARCHIVOS GENERADOS
==========================================================================================
Reportes CSV:
  - data_summary.csv
  - missing_values.csv
  - outliers_iqr.csv
  - user_activity.csv
  - movie_activity.csv

Gráficos PNG:
  - distribution_ratings.png
  - distribution_tag_frequency.png
  - movie_genres.png
  - outliers_tag_frequency.png
  - ratings_by_year.png
  - relationship_user_activity.png
  - tags_by_year.png
  - top_20_tags.png
  - top_tags.png

Referencia del dataset:
F. Maxwell Harper y Joseph A. Konstan, 2015.
The MovieLens Datasets: History and Context.
==========================================================================================